#Langchain Example

## 환경 설정

### Package 설치

In [ ]:
!pip install langchain-openai langchain_google_genai langchain_tavily

### API Key

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path="/content/.env", override=True)

print(".env 내 OPENAI_API_KEY가 환경변수에 할당됐습니다:", os.environ["OPENAI_API_KEY"][:5]+"*****")
print(".env 내 GOOGLE_API_KEY가 환경변수에 할당됐습니다:", os.environ["GOOGLE_API_KEY"][:5]+"*****")
print(".env 내 TAVILY_API_KEY가 환경변수에 할당됐습니다:", os.environ["TAVILY_API_KEY"][:5]+"*****")


### Langchain Component Examples

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "gpt-4o-mini",
    model_provider="openai",
    temperature=0.7
)

response = model.invoke("LangChain의 장점을 설명해줘")
print(response.content)


In [ ]:
# OpenAI 모델
model_openai = init_chat_model("gpt-4o-mini", model_provider="openai")

# Google 모델
model_google = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai")

for m in [model_openai, model_google]:
    print(m.invoke("한 문장으로 자기소개 해줘").content + "\n\n")


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="당신은 친절한 프로그래밍 튜터입니다."),
    HumanMessage(content="파이썬 리스트와 튜플의 차이를 알려줘")
]

response = model.invoke(messages)
print(response.content)


In [ ]:
conversation = []
conversation.append(SystemMessage(content="당신은 요리 전문가입니다."))
conversation.append(HumanMessage(content="김치찌개 레시피 알려줘"))

response = model.invoke(conversation)
conversation.append(response)

conversation.append(HumanMessage(content="더 맵게 하려면?"))
response2 = model.invoke(conversation)
print(response2.content)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 전문가입니다. 친절하고 간결하게 답변하세요."),
    ("human", "{question}")
])

formatted = prompt.invoke({"role": "영양", "question": "다이어트에 좋은 저녁 메뉴 추천해줘"})
print(formatted.to_messages())


[SystemMessage(content='당신은 영양 전문가입니다. 친절하고 간결하게 답변하세요.', additional_kwargs={}, response_metadata={}), HumanMessage(content='다이어트에 좋은 저녁 메뉴 추천해줘', additional_kwargs={}, response_metadata={})]


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

checkpointer = InMemorySaver()
agent = create_agent(model=model, tools=[], checkpointer=checkpointer)

config = {"configurable": {"thread_id": "session-1"}}

agent.invoke({"messages": [{"role": "user", "content": "내 이름은 민준이야"}]}, config)
result = agent.invoke({"messages": [{"role": "user", "content": "내 이름이 뭐라고 했지?"}]}, config)
print(result["messages"][-1].content)


In [ ]:
for chunk in model.stream("인공지능의 역사를 3문단으로 설명해줘"):
    print(chunk.content, end="", flush=True)


In [ ]:
async for chunk in model.astream("파이썬의 장점을 알려줘"):
    print(chunk.content, end="")

In [ ]:
from pydantic import BaseModel

class PersonInfo(BaseModel):
    name: str
    age: int
    job: str

structured_model = model.with_structured_output(PersonInfo)
result = structured_model.invoke(
    "홍길동은 28살이고 개발자로 일하고 있습니다. 정보를 추출해줘"
)

print(result)
print(result.name, result.age, result.job)

In [ ]:
from pydantic import BaseModel, Field

class MovieReview(BaseModel):
    title: str = Field(description="영화 제목")
    rating: int = Field(description="1~5점 사이의 평점")
    summary: str = Field(description="한 줄 감상평")

In [ ]:
structured_model = model.with_structured_output(MovieReview)

result = structured_model.invoke(
    "인터스텔라는 정말 감동적이었어요. 5점 만점에 5점 주고 싶어요"
)

print(result)
print(result.title, result.rating, result.summary)

In [ ]:
from pydantic import Field

class MovieReview(BaseModel):
    rating: int = Field(ge=1, le=5, description="1~5점 사이의 평점")


In [ ]:
class Actor(BaseModel):
    name: str
    role: str

class MovieInfo(BaseModel):
    title: str
    actors: list[Actor]


In [ ]:
{
  "name": "홍길동",
  "age": 28,
  "skills": ["Python", "LangChain"],
  "is_student": False
}

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
prompt = ChatPromptTemplate.from_messages([
    ("human", "다음의 정보로부터 이름, 나이, 직업을 json 형식으로 추출해줘. {text}")
])

chain = prompt | model | parser

result = chain.invoke({"text": "홍길동, 28세, 개발자입니다"})
print(type(result))   # <class 'dict'>
print(result)
print(result["이름"])


In [ ]:
{
  "title": "MovieReview",
  "type": "object",
  "properties": {
    "title": {"type": "string", "description": "영화 제목"},
    "rating": {"type": "integer", "description": "1~5점 평점"}
  },
  "required": ["title", "rating"]
}

In [ ]:
schema = MovieReview.model_json_schema()
print(schema)

# LangChain 내부적으로 이 schema가 Function Calling 형태로 모델에 전달됨
structured_model = model.with_structured_output(MovieReview)

```
{'properties':
  {'rating':
    {'description': '1~5점 사이의 평점', 'maximum': 5,
    'minimum': 1,
    'title': 'Rating',
    'type': 'integer'
    }
  }, 'required': ['rating'],
  'title': 'MovieReview',
  'type': 'object'
}
```

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | model | StrOutputParser()

result = chain.invoke({"role": "여행", "question": "3박 4일 제주 일정 짜줘"})
print(result)


In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini")

def get_weather(city: str) -> str:
    """도시의 날씨 정보를 반환합니다."""
    return f"{city}는 현재 맑음, 22도입니다."

agent = create_agent(model=model, tools=[get_weather])
result = agent.invoke({"messages": [{"role": "user", "content": "서울 날씨 알려줘"}]})
print(result["messages"][-1].content)

In [ ]:
from langchain_core.tools import tool

@tool
def add_numbers(a: int, b: int) -> int:
    """두 정수를 더한 값을 반환합니다."""
    return a + b

In [ ]:
model_with_tools = model.bind_tools([add_numbers])
response = model_with_tools.invoke("3과 5를 더하면?")
print(response.tool_calls)

In [ ]:
from langchain_tavily import TavilySearch

search_tool = TavilySearch(max_results=3)

agent = create_agent(model=model, tools=[search_tool, add_numbers])
result = agent.invoke({
    "messages": [{"role": "user", "content": "오늘 서울 날씨와 12+30을 알려줘"}]
})
print(result["messages"][-1].content)

In [ ]:
from langchain.chat_models import init_chat_model

gpt_4o_mini = init_chat_model("openai:gpt-4o-mini")
gemini = init_chat_model("google_genai:gemini-2.5-flash-lite")

for m in [gpt_4o_mini, gemini]:
    print(m.invoke("오늘 기분이 어때?").content + "\n\n")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("{topic}에 대해 한 문장으로 설명해줘")
parser = StrOutputParser()

chain = prompt | model | parser
result = chain.invoke({"topic": "머신러닝"})
print(result)


In [ ]:
debug_chain = prompt | model
print(debug_chain.invoke({"topic": "딥러닝"}))


In [ ]:
from langchain_core.runnables import RunnableLambda

def to_upper(text: str) -> str:
    return text.upper()

prompt = ChatPromptTemplate.from_template("Explain about the {topic} for elementary school student.")

chain = prompt | model | parser | RunnableLambda(to_upper)
result = chain.invoke({"topic": "Machine Learning"})

print(result)

In [ ]:
from langchain_core.runnables import RunnableSequence
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("{topic}에 대해 한 문장으로 설명해줘")
parser = StrOutputParser()

#chain = prompt | model | parser
chain = RunnableSequence(prompt, model, parser)
result = chain.invoke({"topic": "머신러닝"})

print(result)

In [ ]:
summarize_prompt = ChatPromptTemplate.from_template("다음 글을 요약해줘: {text}")
translate_prompt = ChatPromptTemplate.from_template("다음을 영어로 번역해줘: {summary}")

summarize_chain = summarize_prompt | model | parser
translate_chain = translate_prompt | model | parser

full_chain = (
    {"summary": summarize_chain}
    | translate_chain
)
result = full_chain.invoke({"text": """미국에서 50명 이하의 소규모 하객만 초대하는 '마이크로 웨딩' 트렌드가 확산하고 있다. 치솟는 물가와 젊은 세대의 변화된 가치관이 맞물리며 전통적인 대규모 결혼식 대신 의미 있는 소통을 우선하는 결혼 문화가 자리 잡고 있다.
최근 아시아경제가 인용한 미국 매체 코스모폴리탄은 마이크로 웨딩이 새로운 결혼식 트렌드로 부상했다고 전했다. 매체는 '결혼식 계획에서 가장 핵심적인 결정은 당일 몇 명의 하객을 부를지를 정하는 것'이라며 이같은 변화를 조명했다.
마이크로 웨딩은 명확한 기준은 없으나 통상 하객 50명 미만의 결혼식을 뜻한다. 미국 결혼 정보 사이트 더 노트가 발표한 자료에 따르면 지난해 미국 결혼식의 평균 하객 수는 117명이었다. 마이크로 웨딩은 이보다 절반 이상 줄인 규모다."""})
print(result)

In the U.S., 'micro weddings'—inviting fewer than 50 guests—are becoming popular, reflecting rising living costs and changing values among young generations. This indicates a retreat from traditional large-scale weddings and represents a wedding culture that emphasizes meaningful communication. Last year, the average number of guests at American weddings was 117, showing that micro weddings constitute a size reduction of more than half.


In [ ]:
from langchain_core.runnables import RunnableParallel

summarize_prompt = ChatPromptTemplate.from_template("다음 글을 요약해줘: {text}")
keyword_prompt = ChatPromptTemplate.from_template("다음 글에서 핵심 키워드를 추출해줘: {text}")

parallel_chain = RunnableParallel(
    summary=summarize_prompt | model | parser,
    keywords=keyword_prompt | model | parser
)

result = parallel_chain.invoke({"text": """미국에서 50명 이하의 소규모 하객만 초대하는 '마이크로 웨딩' 트렌드가 확산하고 있다. 치솟는 물가와 젊은 세대의 변화된 가치관이 맞물리며 전통적인 대규모 결혼식 대신 의미 있는 소통을 우선하는 결혼 문화가 자리 잡고 있다.
최근 아시아경제가 인용한 미국 매체 코스모폴리탄은 마이크로 웨딩이 새로운 결혼식 트렌드로 부상했다고 전했다. 매체는 '결혼식 계획에서 가장 핵심적인 결정은 당일 몇 명의 하객을 부를지를 정하는 것'이라며 이같은 변화를 조명했다.
마이크로 웨딩은 명확한 기준은 없으나 통상 하객 50명 미만의 결혼식을 뜻한다. 미국 결혼 정보 사이트 더 노트가 발표한 자료에 따르면 지난해 미국 결혼식의 평균 하객 수는 117명이었다. 마이크로 웨딩은 이보다 절반 이상 줄인 규모다."""})
print(result)
print(result["summary"])
print(result["keywords"])

{'summary': "미국에서는 젊은 세대의 변화된 가치관과 물가 상승을 고려해 50명 미만의 소규모 하객을 부르는 '마이크로 웨딩'이 유행하고 있다. 이는 전통적인 대규모 결혼식 대신 의미 있는 소통을 중시하는 결혼 문화의 변화를 반영한 것으로, 이를 보고한 미국 매체 코스모폴리탄은 결혼식 계획에서 가장 중요한 것이 하객 수를 결정하는 것이라고 전했다.", 'keywords': "미국, 50명, 소규모 하객, '마이크로 웨딩', 트렌드, 확산, 물가, 젊은 세대, 가치관, 대규모 결혼식, 소통, 결혼 문화, 아시아경제, 코스모폴리탄, 결혼식 계획, 변화, 하객 50명 미만의 결혼식, 더 노트, 자료, 결혼식 평균 하객 수, 117명."}

미국에서는 젊은 세대의 변화된 가치관과 물가 상승을 고려해 50명 미만의 소규모 하객을 부르는 '마이크로 웨딩'이 유행하고 있다. 이는 전통적인 대규모 결혼식 대신 의미 있는 소통을 중시하는 결혼 문화의 변화를 반영한 것으로, 이를 보고한 미국 매체 코스모폴리탄은 결혼식 계획에서 가장 중요한 것이 하객 수를 결정하는 것이라고 전했다.

미국, 50명, 소규모 하객, '마이크로 웨딩', 트렌드, 확산, 물가, 젊은 세대, 가치관, 대규모 결혼식, 소통, 결혼 문화, 아시아경제, 코스모폴리탄, 결혼식 계획, 변화, 하객 50명 미만의 결혼식, 더 노트, 자료, 결혼식 평균 하객 수, 117명.


### Gradio UI Component

In [ ]:
import gradio as gr

def chatbot_response(message, history):
    response = agent.invoke({"messages": [{"role": "user", "content": message}]})
    return response["messages"][-1].content

demo = gr.ChatInterface(fn=chatbot_response)
demo.launch()

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## LangChain 기반 문서 요약기")
    input_text = gr.Textbox(label="원문 입력")
    output_text = gr.Textbox(label="요약 결과")
    btn = gr.Button("요약하기")
    btn.click(fn=summarize_chain.invoke, inputs=input_text, outputs=output_text)

demo.launch()
